In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW_PROCESSED_PATH = "../data/processed/"

df_monthly = pd.read_csv(RAW_PROCESSED_PATH + "sales_monthly_clean.csv")

df_monthly['datum'] = pd.to_datetime(df_monthly['datum'])
df_monthly.set_index('datum', inplace = True)

drug_cols = ['m01ab', 'm01ae', 'n02ba', 'n02be', 'n05b', 'n05c', 'r03', 'r06']
df_monthly['total_sales'] = df_monthly[drug_cols].sum(axis=1)

df_daily = pd.read_csv(RAW_PROCESSED_PATH + "sales_daily_clean.csv")

df_daily['datum'] = pd.to_datetime(df_daily['datum'])
df_daily.set_index('datum', inplace = True)

df_daily['total_sales'] = df_daily[drug_cols].sum(axis= 1)

In [24]:
#LAG Feature for months

df_monthly['lag_1'] = df_monthly['total_sales'].shift(1)
df_monthly['lag_2'] = df_monthly['total_sales'].shift(2)
df_monthly['lag_3'] = df_monthly['total_sales'].shift(3)
df_monthly['lag_12'] = df_monthly['total_sales'].shift(12)

print(f"Lag feature :\n{df_monthly[['total_sales', 'lag_1', 'lag_2', 'lag_3', 'lag_12']].head(15)}")

Lag feature :
            total_sales     lag_1     lag_2     lag_3   lag_12
datum                                                         
2014-01-31     1821.110       NaN       NaN       NaN      NaN
2014-02-28     1974.470  1821.110       NaN       NaN      NaN
2014-03-31     1606.720  1974.470  1821.110       NaN      NaN
2014-04-30     1429.675  1606.720  1974.470  1821.110      NaN
2014-05-31     1506.303  1429.675  1606.720  1974.470      NaN
2014-06-30     1390.205  1506.303  1429.675  1606.720      NaN
2014-07-31     1332.370  1390.205  1506.303  1429.675      NaN
2014-08-31     1499.020  1332.370  1390.205  1506.303      NaN
2014-09-30     1814.594  1499.020  1332.370  1390.205      NaN
2014-10-31     3146.906  1814.594  1499.020  1332.370      NaN
2014-11-30     1770.640  3146.906  1814.594  1499.020      NaN
2014-12-31     2093.485  1770.640  3146.906  1814.594      NaN
2015-01-31     2157.749  2093.485  1770.640  3146.906  1821.11
2015-02-28     1831.532  2157.749  2093.4

In [25]:
#rolling window

df_monthly['rolling_mean_3'] = df_monthly['total_sales'].rolling(window=3).mean()
df_monthly['rolling_mean_6'] = df_monthly['total_sales'].rolling(window=6).mean()
df_monthly['rolling_std_3'] = df_monthly['total_sales'].rolling(window=3).std()

df_monthly[['total_sales', 'rolling_mean_3', 'rolling_mean_6', 'rolling_std_3']].iloc[6:20]

,total_sales,rolling_mean_3,rolling_mean_6,rolling_std_3
datum,,,,
2014-07-31,1332.370,1409.626000,1539.957167,88.577952
2014-08-31,1499.020,1407.198333,1460.715500,84.614630
2014-09-30,1814.594,1548.661333,1495.361167,244.914665
2014-10-31,3146.906,2153.506667,1781.566333,874.659027
2014-11-30,1770.640,2244.046667,1825.622500,782.207914
2014-12-31,2093.485,2337.010333,1942.835833,719.725966
2015-01-31,2157.749,2007.291333,2080.399000,207.449646
2015-02-28,1831.532,2027.588667,2135.817667,172.803726
2015-03-31,1974.076,1987.785667,2162.398000,163.540053


In [26]:
#cyclical encoding using sine and cosine

df_monthly['month'] = df_monthly.index.month

df_monthly['month_sin'] = np.sin(2 * np.pi * df_monthly['month'] / 12)
df_monthly['month_cos'] = np.cos(2 * np.pi * df_monthly['month'] / 12)


In [27]:
#verification code

test_dates = ['2014-12-31', '2014-06-30', '2015-01-31']

print("month sine values")
print(df_monthly.loc[test_dates, 'month_sin'])

print("month cosine values")
print(df_monthly.loc[test_dates, 'month_cos'])

month sine values
datum
2014-12-31   -2.449294e-16
2014-06-30    1.224647e-16
2015-01-31    5.000000e-01
Name: month_sin, dtype: float64
month cosine values
datum
2014-12-31    1.000000
2014-06-30   -1.000000
2015-01-31    0.866025
Name: month_cos, dtype: float64


In [37]:
#LAG features for daily dataset

df_daily['total_sales'] = df_daily[drug_cols].sum(axis= 1)

df_daily['lag_1'] = df_daily['total_sales'].shift(1)  #yesterday
df_daily['lag_7'] = df_daily['total_sales'].shift(7)  #same day as last week

print(f"Lag feature: \n{df_daily[['total_sales', 'lag_1', 'lag_7']].head(20)}")


#rolling window
df_daily['rolling_mean_7'] = df_daily['total_sales'].rolling(window=7).mean()
df_daily['rolling_std_7'] = df_daily['total_sales'].rolling(window=7).std()

df_daily[['total_sales', 'rolling_mean_7', 'rolling_std_7']].iloc[7:20]


Lag feature: 
            total_sales   lag_1   lag_7
datum                                  
2014-01-02        48.47     NaN     NaN
2014-01-03       107.00   48.47     NaN
2014-01-04        91.35  107.00     NaN
2014-01-05        66.10   91.35     NaN
2014-01-06        58.20   66.10     NaN
2014-01-07         0.00   58.20     NaN
2014-01-08        75.23    0.00     NaN
2014-01-09        62.68   75.23   48.47
2014-01-10        81.30   62.68  107.00
2014-01-11        87.24   81.30   91.35
2014-01-12        27.16   87.24   66.10
2014-01-13        90.20   27.16   58.20
2014-01-14        62.33   90.20    0.00
2014-01-15        58.04   62.33   75.23
2014-01-16        65.60   58.04   62.68
2014-01-17        58.38   65.60   81.30
2014-01-18        83.33   58.38   87.24
2014-01-19        32.43   83.33   27.16
2014-01-20        59.54   32.43   90.20
2014-01-21        69.34   59.54   62.33


,total_sales,rolling_mean_7,rolling_std_7
datum,,,
2014-01-09,62.68,65.794286,33.754979
2014-01-10,81.30,62.122857,29.677579
2014-01-11,87.24,61.535714,29.036712
2014-01-12,27.16,55.972857,31.630743
2014-01-13,90.20,60.544286,34.213240
2014-01-14,62.33,69.448571,21.625020
2014-01-15,58.04,66.992857,21.834093
2014-01-16,65.60,67.410000,21.765749
2014-01-17,58.38,64.135714,21.039840


In [43]:
#cyclical encoding for daily dataset using sine and cosine

df_daily['day_of_week'] = df_daily.index.dayofweek

df_daily['dow_sin'] = np.sin(2 * np.pi * df_daily['day_of_week'] / 7)
df_daily['dow_cos'] = np.cos(2 * np.pi * df_daily['day_of_week'] / 7)

In [40]:
#verification code

daily_test_dates = ['2014-01-05', '2014-01-06']

print("daily sine values")
print(df_daily.loc[daily_test_dates, 'dow_sin'])

print("daily cosine values")
print(df_daily.loc[daily_test_dates, 'dow_cos'])

daily sine values
datum
2014-01-05   -0.781831
2014-01-06    0.000000
Name: dow_sin, dtype: float64
daily cosine values
datum
2014-01-05    0.62349
2014-01-06    1.00000
Name: dow_cos, dtype: float64


In [41]:
print(df_daily.loc[['2014-01-06','2014-01-09'], ['day_of_week','dow_sin','dow_cos']])

            day_of_week   dow_sin   dow_cos
datum                                      
2014-01-06            0  0.000000  1.000000
2014-01-09            3  0.433884 -0.900969


In [42]:
RAW_PROCESSED_PATH = "../data/processed/"

filepath = RAW_PROCESSED_PATH + 'sales_monthly_lag.csv'
df_monthly.to_csv(filepath, index = True)

print(f"Monthly lag data saved at: {filepath}")

filepath1 = RAW_PROCESSED_PATH + 'sales_daily_lag.csv'
df_daily.to_csv(filepath1, index = True)
print(f"Daily lag data saved at: {filepath1}")

Monthly lag data saved at: ../data/processed/sales_monthly_lag.csv
Daily lag data saved at: ../data/processed/sales_daily_lag.csv
